In [138]:
# imports

import pandas as pd
import numpy as np
import json
import re
import string
from collections import Counter
from sentence_transformers import SentenceTransformer, util

c:\Users\aca23lr\source\dissertation\tabfact_data_analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Helpers to read in data and convert to dataframe

def load_json_data(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

def convert_to_dataframe(data, is_full_cleaned=False):
    record = []
    for table_id, details in data.items():
        statements = details[0]
        enatiled_statuses = details[1]
        caption = details[-1]
        
        if is_full_cleaned:
            pos_tags = details[2]
            for statement, entailed_status, statement_pos_tags in zip(statements, enatiled_statuses, pos_tags):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "entailed_status": entailed_status,
                    "caption": caption,
                    "pos_tags": statement_pos_tags
                }) 
        else:
            for statement, entailed_status in zip(statements, enatiled_statuses):
                record.append({
                    "table_id": table_id,
                    "statement": statement,
                    "entailed_status": entailed_status,
                    "caption": caption
                })
    return pd.DataFrame(record)

def load_and_convert_data_to_dataframe(path, is_full_cleaned=False):
    raw_data = load_json_data(path)
    
    return convert_to_dataframe(raw_data, is_full_cleaned=is_full_cleaned)

In [3]:
# Load in data frames

df_full_cleaned = load_and_convert_data_to_dataframe("data/full_cleaned.json", is_full_cleaned=True)

df_full_cleaned_refuted = df_full_cleaned[df_full_cleaned['entailed_status'] == 0]
df_full_cleaned_entailed = df_full_cleaned[df_full_cleaned['entailed_status'] == 1]

df_r1_training = load_and_convert_data_to_dataframe("data/r1_training_all.json")

df_r2_training = load_and_convert_data_to_dataframe("data/r2_training_all.json")

df_training = load_and_convert_data_to_dataframe("data/train_examples.json")

df_test = load_and_convert_data_to_dataframe("data/test_examples.json")

df_validation = load_and_convert_data_to_dataframe("data/val_examples.json")


In [4]:
# Replicate their simple and complex channel data analysis: gathering

# Identify unique tables
tables_in_full_cleaned = df_full_cleaned["table_id"].unique()
tables_in_r1 = df_r1_training["table_id"].unique()
tables_in_r2 = df_r2_training["table_id"].unique()

# Find tables that are in r1/r2 and in full cleaned
tables_in_r1_and_full_cleaned = np.intersect1d(tables_in_r1, tables_in_full_cleaned)
tables_in_r2_and_full_cleaned = np.intersect1d(tables_in_r2, tables_in_full_cleaned)

# Identify r1 and r2 statements in full cleaned
r1_in_full_cleaned = df_full_cleaned[df_full_cleaned["table_id"].isin(tables_in_r1_and_full_cleaned)]
r2_in_full_cleaned = df_full_cleaned[df_full_cleaned["table_id"].isin(tables_in_r2_and_full_cleaned)]

# Average length of entailed statements
full_cleaned_entailed_statements = df_full_cleaned_entailed["statement"]
average_length_full_cleaned_entailed_statements = full_cleaned_entailed_statements.str.split().apply(len).mean()

full_cleaned_entailed_statements_from_r1_tables = r1_in_full_cleaned[r1_in_full_cleaned["entailed_status"] == 1]["statement"]
average_length_full_cleaned_entailed_statements_from_r1_tables = full_cleaned_entailed_statements_from_r1_tables.str.split().apply(len).mean()

full_cleaned_entailed_statements_from_r2_tables = r2_in_full_cleaned[r2_in_full_cleaned["entailed_status"] == 1]["statement"]
average_length_full_cleaned_entailed_statements_from_r2_tables = full_cleaned_entailed_statements_from_r2_tables.str.split().apply(len).mean()


# Average length of refuted statements
full_cleaned_refuted_statements = df_full_cleaned_refuted["statement"]
average_length_full_cleaned_refuted_statements = full_cleaned_refuted_statements.str.split().apply(len).mean()

full_cleaned_refuted_statements_from_r1_tables = r1_in_full_cleaned[r1_in_full_cleaned["entailed_status"] == 0]["statement"]
average_length_full_cleaned_refuted_statements_from_r1_tables = full_cleaned_refuted_statements_from_r1_tables.str.split().apply(len).mean()

full_cleaned_refuted_statements_from_r2_tables = r2_in_full_cleaned[r2_in_full_cleaned["entailed_status"] == 0]["statement"]
average_length_full_cleaned_refuted_statements_from_r2_tables = full_cleaned_refuted_statements_from_r2_tables.str.split().apply(len).mean()

In [5]:
# Replicate their simple and complex channel data analysis: displaying results 

# Tables
print(f"Number of unique tables in full cleaned dataset: {len(tables_in_full_cleaned)}")
print(f"Number of unique tables in r1 training dataset: {len(tables_in_r1)}")
print(f"Number of unique tables in r2 training dataset: {len(tables_in_r2)}")

# Number of statements
print(f"Number of statements in full cleaned dataset: {len(df_full_cleaned)}")
print(f"Number of statements in r1 training dataset: {len(r1_in_full_cleaned)}")
print(f"Number of statements in r2 training dataset: {len(r2_in_full_cleaned)}")

# Statement length of entailed statements
print(f"Average length of entailed statements in full cleaned dataset: {average_length_full_cleaned_entailed_statements:.2f} words")
print(f"Average length of entailed statements in r1 training dataset: {average_length_full_cleaned_entailed_statements_from_r1_tables:.2f} words")
print(f"Average length of entailed statements in r2 training dataset: {average_length_full_cleaned_entailed_statements_from_r2_tables:.2f} words")

# Statement length of refuted statements
print(f"Average length of refuted statements in full cleaned dataset: {average_length_full_cleaned_refuted_statements:.2f} words")
print(f"Average length of refuted statements in r1 training dataset: {average_length_full_cleaned_refuted_statements_from_r1_tables:.2f} words")
print(f"Average length of refuted statements in r2 training dataset: {average_length_full_cleaned_refuted_statements_from_r2_tables:.2f} words")


Number of unique tables in full cleaned dataset: 16573
Number of unique tables in r1 training dataset: 9181
Number of unique tables in r2 training dataset: 7392
Number of statements in full cleaned dataset: 117854
Number of statements in r1 training dataset: 49850
Number of statements in r2 training dataset: 68004
Average length of entailed statements in full cleaned dataset: 13.71 words
Average length of entailed statements in r1 training dataset: 13.05 words
Average length of entailed statements in r2 training dataset: 14.19 words
Average length of refuted statements in full cleaned dataset: 13.79 words
Average length of refuted statements in r1 training dataset: 13.32 words
Average length of refuted statements in r2 training dataset: 14.15 words


In [6]:
# Checking number of satetements in training, test, and validation datasets

print(f"Number of statements in training dataset: {len(df_training)}")
print(f"Number of statements in test dataset: {len(df_test)}")
print(f"Number of statements in validation dataset: {len(df_validation)}")
print(f"Total number of statements in training, test, and validation datasets: {len(df_training) + len(df_test) + len(df_validation)}")


Number of statements in training dataset: 92283
Number of statements in test dataset: 12779
Number of statements in validation dataset: 12792
Total number of statements in training, test, and validation datasets: 117854


In [7]:
# Entity analysis

# Counting average ENT tags per statement
average_entities_per_statement_full_cleaned = df_full_cleaned['pos_tags'].str.count(r'\bENT\b').mean()
average_entities_per_entailed_statement = df_full_cleaned_entailed['pos_tags'].str.count(r'\bENT\b').mean()
average_entities_per_refuted_statement = df_full_cleaned_refuted['pos_tags'].str.count(r'\bENT\b').mean()

print(f"Average number of ENT tags per statement in full cleaned dataset: {average_entities_per_statement_full_cleaned:.2f}")
print(f"Average number of ENT tags per entailed statement: {average_entities_per_entailed_statement:.2f}")
print(f"Average number of ENT tags per refuted statement: {average_entities_per_refuted_statement:.2f}")

# Counting average content words
content_word_regex = r'\b(ENT|NN|NNS|NNP|NNPS|VB|VBD|VBG|VBN|VBP|VBZ|JJ|JJR|JJS|RB|RBR|RBS|CD)\b'
average_content_words_per_statement_full_cleaned = df_full_cleaned['pos_tags'].str.count(content_word_regex).mean()
average_content_words_per_entailed_statement = df_full_cleaned_entailed['pos_tags'].str.count(content_word_regex).mean()
average_content_words_per_refuted_statement = df_full_cleaned_refuted['pos_tags'].str.count(content_word_regex).mean()
average_content_words_per_statement_r1 = r1_in_full_cleaned['pos_tags'].str.count(content_word_regex).mean()
average_content_words_per_statement_r2 = r2_in_full_cleaned['pos_tags'].str.count(content_word_regex).mean()

print(f"Average number of content words per statement in full cleaned dataset: {average_content_words_per_statement_full_cleaned:.2f}")
print(f"Average number of content words per entailed statement: {average_content_words_per_entailed_statement:.2f}")
print(f"Average number of content words per refuted statement: {average_content_words_per_refuted_statement:.2f}")
print(f"Average number of content words per statement in r1 training dataset: {average_content_words_per_statement_r1:.2f}")
print(f"Average number of content words per statement in r2 training dataset: {average_content_words_per_statement_r2:.2f}")

Average number of ENT tags per statement in full cleaned dataset: 3.49
Average number of ENT tags per entailed statement: 3.51
Average number of ENT tags per refuted statement: 3.46
Average number of content words per statement in full cleaned dataset: 7.15
Average number of content words per entailed statement: 7.08
Average number of content words per refuted statement: 7.23
Average number of content words per statement in r1 training dataset: 6.50
Average number of content words per statement in r2 training dataset: 7.62


In [12]:
# Calculating word counts

# Kind of useless
def get_non_entity_statement_word_counts(df):
    statements_without_entities = (df['statement']
                                   .str.replace(r'#.*?#', '', regex=True) # get rid of entiteis
                                   .str.lower()
                                   .str.replace(rf'[{re.escape(string.punctuation)}]', ' ', regex=True)) # get rid of punctuation
    word_counts = statements_without_entities.str.split().explode().value_counts() # get words into single list of words and count them
    return word_counts

# More useful
def get_non_entity_statement_content_words(df):
    statements_without_entities = (df['statement']
                                   .str.replace(r'#.*?#', '', regex=True) # get rid of entities
                                   .str.lower()
                                   .str.split())
    pos_tags_without_entities = df['pos_tags'].str.replace(r'\bENT\b', '', regex=True).str.split() # get rid of ENT tags
    for statement_list, pos_tag_list in zip(statements_without_entities, pos_tags_without_entities):
        content_words = [word for word, pos in zip(statement_list, pos_tag_list) if re.match(content_word_regex, pos)]
        yield from content_words # more memory efficient than returning a list of all content words

full_cleaned_content_word_counts = Counter(get_non_entity_statement_content_words(df_full_cleaned))
print(full_cleaned_content_word_counts.most_common(20))

full_cleaned_entailed_content_word_counts = Counter(get_non_entity_statement_content_words(df_full_cleaned_entailed))
print(full_cleaned_entailed_content_word_counts.most_common(20))

full_cleaned_refuted_content_word_counts = Counter(get_non_entity_statement_content_words(df_full_cleaned_refuted))
print(full_cleaned_refuted_content_word_counts.most_common(20))

[('be', 78143), ('have', 28405), ('game', 15298), ('play', 10075), ('only', 8451), ('2', 8051), ('1', 7684), ('more', 7546), ('time', 6665), ('win', 6661), ('team', 6543), ('number', 5144), ('3', 5033), ('highest', 4799), ('most', 3719), ('4', 3210), ('season', 2940), ('less', 2937), ('not', 2923), ('point', 2794)]
[('be', 41481), ('have', 15391), ('game', 8260), ('play', 5389), ('only', 4523), ('2', 4467), ('1', 4226), ('win', 3609), ('time', 3597), ('more', 3595), ('team', 3504), ('number', 2699), ('3', 2667), ('highest', 2597), ('most', 2147), ('4', 1731), ('season', 1537), ('episode', 1509), ('point', 1502), ('score', 1498)]
[('be', 36662), ('have', 13014), ('game', 7038), ('play', 4686), ('more', 3951), ('only', 3928), ('2', 3584), ('1', 3458), ('time', 3068), ('win', 3052), ('team', 3039), ('number', 2445), ('3', 2366), ('highest', 2202), ('less', 1983), ('most', 1572), ('4', 1479), ('not', 1443), ('season', 1403), ('lowest', 1401)]


In [ ]:
# Average entity word length

def get_entities(df):
    entities = df['statement'].str.findall(r'#.*?#')
    return entities.explode().dropna() # remove rows that had no hashtags

entities = get_entities(df_full_cleaned)
entity_words = entities.str[1:-1].str.extract(r"^(.*);", expand=False)
average_entity_word_length = entity_words.str.split().apply(len).mean()
print(f"Average entity word length is: {average_entity_word_length:.2f}")

Average entity word length is: 1.72


In [ ]:
# Rows and columns identified per statement

def count_rows(entity_arr):
    return len({coord.split(",")[0] for coord in entity_arr})

def count_columns(entity_arr):
    return len({coord.split(",")[1] for coord in entity_arr})

def get_rows_columns_mentioned_by_entities(entity_arr):
    return [re.search(r";([^;]*)$", entity[1:-1]).group(1) for entity in entity_arr]

entity_cell_coordinates = df_full_cleaned['statement'].str.findall(r'#.*?#').dropna().apply(get_rows_columns_mentioned_by_entities)

average_cell_coords_identified_per_statement = entity_cell_coordinates.apply(len).mean()
average_row_count_identified_per_statement = entity_cell_coordinates.apply(count_rows).mean()
average_col_count_identified_per_statement = entity_cell_coordinates.apply(count_columns).mean()

print(f"Average cell coordinate evidence identified per statement is: {average_cell_coords_identified_per_statement:.2f}")
print(f"Average rows looked at per statement: {average_row_count_identified_per_statement:.2f}")
print(f"Average columns looked at per statement: {average_col_count_identified_per_statement:.2f}")

Average cell coordinate evidence identified per statement is: 3.49
Average rows looked at per statement: 2.57
Average columns looked at per statement: 2.37


In [ ]:
# Average cosine similarity with the caption

allMiniModel = SentenceTransformer("all-MiniLM-L6-v2")

def compute_cosine_similarity(caption, statement):
    caption_emb = allMiniModel.encode(caption, convert_to_tensor=True)
    statement_emb = allMiniModel.encode(statement, convert_to_tensor=True)
    return util.cos_sim(caption_emb, statement_emb).item()

# # Compute embeddings
# emb_caption = model.encode(caption, convert_to_tensor=True)
# emb_statement = model.encode(statement, convert_to_tensor=True)

# # Compute Cosine Similarity
# cosine_sim = util.cos_sim(emb_caption, emb_statement).item()
# print(cosine_sim)  # Yields a score between -1 and 1